# SubspaceAD (`subspacead-final-v1`) — Official Visual Wrap-Up, `672×672` Full-Res Eval & DefectLens Implementation Assets (CPU-Only)

This companion notebook is built specifically for your **official `defectlens-subspacead-final`** Kaggle Dataset (`SubspaceAD-Official-Final`). It uses **0 GPU hours** and **0 backbone forward passes**.

### What This Notebook Produces (Bundled in `subspacead_official_assets.zip`)
1. **Publication-Ready Benchmark & Calibration Charts (`figures/`, `300 DPI` PNGs):**
   * `fig1_fewshot_scaling.png` — Few-shot scaling (`1 → 2 → 4-shot`, 5-seed mean ± std) vs Regime `T` (0-shot) & Official CVPR 2026 README (`97.99%` MVTec / `94.75%` VisA), with clean non-overlapping labels.
   * `fig2_mvtec_per_category_R4_vs_T.png` & `fig2_visa_per_category_R4_vs_T.png` — Per-category `4-shot` vs `0-shot` comparison (`Image AUROC` & `AUPRO@30%`).
   * `fig3_score_distributions_calibration.png` — Normal vs Anomalous raw score histograms (`bottle`, `capsule`, `transistor`, `pcb3`).
   * `fig4_regime_scale_mismatch.png` — Cross-category & cross-regime raw score scale analysis (proving why per-category Calibration / Conformal Prediction is required in DefectLens).
   * `fig5_vlm_roi_crop_preview.png` — Visual verification of automatically extracted ROI bounding boxes & zoomed crops for the DefectLens VLM Verifier (Qwen-VL).
2. **Qualitative Heatmaps & Failure Analysis (`heatmaps/`, `300 DPI` PNGs):**
   * 7 diagnostic sheets (`bottle`, `capsule`, `transistor`, `screw`, `pcb3`, `macaroni2`, `cashew`): Original | GT Contour | Regime `R` (4-shot, s0) | Regime `T` (0-shot), including **Hardest Defect (Min Score / FN)** and **Hardest Normal (Max FP)** with honest category-wide percentile color scaling.
3. **DefectLens Implementation-Phase Artifacts (`implementation/`):**
   * `subspacead_calibration_stats.csv` — Per-category / per-seed Normal & Anomalous score distribution statistics, quantiles (`q90`, `q95`, `q99`), separation gaps, and operating thresholds (`FPR@1%`, `FPR@5%`, `Oracle-F1`) ready for Conformal Prediction.
   * `subspacead_vlm_roi_proposals.csv` — Per-image peak-anomaly ROI bounding boxes (`x1, y1, x2, y2` in `672px` and normalized `[0, 1]` coordinates), standardized anomaly z-scores, and triage routing bands (`auto_normal` / `borderline_send_to_vlm` / `high_conf_anomaly`) across all 27 categories (`R 4-shot s0` & `T 0-shot`).
4. **Official Full-Resolution `EVAL_SIZE = 672` Pixel Re-Evaluation (`eval672/`, `0` GPU forward passes):**
   * Exact `pix_auroc`, `pix_ap`, `pix_f1_max_oracle`, `aupro_30`, and `aupro_5` at official `672×672` directly from cached `maps.npy` + `gt/<ds>/<cat>/masks.npz` (all 27 categories), plus `eval256_vs_eval672_comparison.csv`.

### Kaggle Settings (0 GPU Quota Used!)
* **Accelerator:** **`None` (CPU)** — uses **0 GPU hours**.
* **Internet:** **`ON`** (needed to auto-download VisA RGB images into `/kaggle/temp/data` for VisA heatmaps & VLM crop previews; `672×672` metrics read cached `gt/masks.npz` directly).
* **Add Input:** Attach **`defectlens-subspacead-final`** (*Your Work -> Datasets*) and **`MVTec AD`** (`ipythonx/mvtec-ad`).


## 1. Configuration & Auto-Discovery (`defectlens-subspacead-final`)

In [ ]:
# ============================ 1. CONFIG & AUTO-DISCOVERY ============================
import os, sys, json, time, zipfile, subprocess, warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from scipy import ndimage
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, roc_curve

warnings.filterwarnings("ignore")

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./work_official_visual")
DATA_CACHE = Path("/kaggle/temp/data") if ON_KAGGLE else (WORK / "data")
OUT_DIR = WORK / "subspacead_official_assets"
FIG_DIR = OUT_DIR / "figures"
HEATMAP_DIR = OUT_DIR / "heatmaps"
IMPL_DIR = OUT_DIR / "implementation"
EVAL672_DIR = OUT_DIR / "eval672"
for d in [WORK, DATA_CACHE, OUT_DIR, FIG_DIR, HEATMAP_DIR, IMPL_DIR, EVAL672_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- User Controls ---
IMG_SIZE = 672
BLUR_K = 3
BLUR_SIGMA = 4.0
TOP_PCT = 1.0
PRO_N_THR = 300   # Matches official SubspaceAD compute_aupro (main.py)

# Categories for publication-grade diagnostic heatmap sheets:
SHOW_CATS: List[Tuple[str, str]] = [
    ("mvtec", "bottle"),      # Near-perfect reference (100% R & T)
    ("mvtec", "capsule"),     # Missing imprint (logical defect -> motivates VLM)
    ("mvtec", "transistor"),  # Official NO-AUG (+3.35% Img AUROC, +8.15% AUPRO@30 vs rotated)
    ("mvtec", "screw"),       # Fine small defects + background corner FP
    ("visa", "pcb3"),         # Missing LED / industrial PCB (Regime T beats 4-shot)
    ("visa", "macaroni2"),    # Hard multi-instance (+3.89% gain with official PIL.rotate)
    ("visa", "cashew"),       # Strong Regime R localization (98.88%)
]

# Downstream VLM crop preview categories (4 hard industrial/logical categories):
VLM_PREVIEW_CATS: List[Tuple[str, str]] = [
    ("mvtec", "capsule"),
    ("mvtec", "transistor"),
    ("mvtec", "screw"),
    ("visa", "pcb3"),
]

# Full-resolution 672x672 pixel evaluation settings (uses cached gt/masks.npz, 0 GPU forward passes):
RUN_EVAL_672 = True
EVAL_672_SEEDS = "first"     # "first" (~15-25 min CPU: seed 0 for 1/2/4-shot + Regime T across all 27 cats)
                             # or "all" (~60-90 min CPU: all 5 seeds; safe in Save & Run All CPU 12h limit)
AUTO_DOWNLOAD_VISA = True    # Download VisA RGB images into /kaggle/temp/data for VisA heatmaps/VLM preview
VISA_RGB_VISUALS = True      # Note: 672x672 eval & calibration/ROI CSVs run on VisA even if False!

MVTEC_CATS = [
    "bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather",
    "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper",
]
VISA_CATS = [
    "candle", "capsules", "cashew", "chewinggum", "fryum", "macaroni1",
    "macaroni2", "pcb1", "pcb2", "pcb3", "pcb4", "pipe_fryum",
]
CATS: Dict[str, List[str]] = {"mvtec": MVTEC_CATS, "visa": VISA_CATS}

# Official CVPR 2026 README Image AUROC (%)
REF_IMG_AUROC_R = {"mvtec": {1: 97.1, 2: 97.5, 4: 98.0}, "visa": {1: 93.2, 2: 93.8, 4: 94.7}}
REF_IMG_AUROC_T = {"mvtec": 96.6, "visa": 94.1}


def locate_official_bundle(search_bases: Optional[Sequence[str]] = None) -> Dict[str, Optional[Path]]:
    """Auto-discover the exact layout of defectlens-subspacead-final (subspacead-final-v1)."""
    bases = list(search_bases or ["/kaggle/input", "/kaggle/working", "."])
    ckpt_root = None
    for base in bases:
        if not os.path.isdir(base):
            continue
        base_depth = base.rstrip(os.sep).count(os.sep)
        for root, dirs, _ in os.walk(base):
            if root.count(os.sep) - base_depth >= 6:
                dirs[:] = []
                continue
            if os.path.basename(root) == "subspacead_ckpt" and (Path(root) / "full").is_dir():
                full_p = Path(root) / "full"
                # Prefer official final-v1 bundle (contains gt/ and regT_*)
                if (full_p / "gt").is_dir() and list(full_p.glob("regT_*")):
                    ckpt_root = Path(root)
                    break
        if ckpt_root is not None:
            break

    if ckpt_root is None:
        raise FileNotFoundError(
            "Could not find official 'subspacead_ckpt/full' (with gt/ and regT_*). "
            "On Kaggle, click '+ Add Input' -> Your Work -> Datasets -> attach 'defectlens-subspacead-final'."
        )

    full_dir = ckpt_root / "full"
    gt_root = full_dir / "gt"
    feat_dirs = sorted(p for p in full_dir.glob("feat_*") if p.is_dir())
    assert feat_dirs, f"No feat_* directory inside {full_dir}"
    feat_dir = feat_dirs[-1]

    fit_r_dirs = sorted(p for p in feat_dir.glob("fit_*") if p.is_dir())
    fit_t_dirs = sorted(p for p in full_dir.glob("regT_*") if p.is_dir())
    assert fit_r_dirs, f"No fit_* directory inside {feat_dir}"
    assert fit_t_dirs, f"No regT_* directory inside {full_dir}"

    parent = ckpt_root.parent
    res_dir = parent / "results_full"
    all_runs_csv = (res_dir / "all_runs.csv") if (res_dir / "all_runs.csv").exists() else None
    per_item_all = (res_dir / "per_item_all.parquet") if (res_dir / "per_item_all.parquet").exists() else None
    if per_item_all is None and (res_dir / "per_item_all.csv").exists():
        per_item_all = res_dir / "per_item_all.csv"

    return dict(
        parent=parent,
        full_dir=full_dir,
        gt_root=gt_root,
        feat_dir=feat_dir,
        fit_r_dir=fit_r_dirs[-1],
        fit_t_dir=fit_t_dirs[-1],
        results_dir=res_dir if res_dir.is_dir() else None,
        all_runs_csv=all_runs_csv,
        per_item_all=per_item_all,
    )


def find_dataset_root(name: str, search: Optional[Sequence[str]] = None, max_depth: int = 5) -> Optional[Path]:
    cats = CATS[name]
    need = 10 if name == "mvtec" else 8
    search = search or ["/kaggle/input", str(DATA_CACHE), "/kaggle/temp", str(WORK), "./data"]
    for base in search:
        if not os.path.isdir(base):
            continue
        base_depth = base.rstrip(os.sep).count(os.sep)
        for root, dirs, _ in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("subspacead_ckpt", "results_full")]
            if root.count(os.sep) - base_depth >= max_depth:
                dirs[:] = []
                continue
            if name == "visa" and "split_csv" in dirs and (Path(root) / "split_csv" / "1cls.csv").exists():
                return Path(root)
            if sum(c in dirs for c in cats) >= need:
                return Path(root)
    return None


def ensure_dataset_opt(name: str) -> Optional[Path]:
    root = find_dataset_root(name)
    if root is not None:
        print(f"[{name}] RGB images found at {root}")
        return root
    if name == "visa" and (not AUTO_DOWNLOAD_VISA or not VISA_RGB_VISUALS):
        print("[visa] RGB images skipped (672x672 eval & calibration/ROI CSVs still run from cached gt/ & maps.npy).")
        return None
    print(f"[{name}] RGB images not attached -> downloading into {DATA_CACHE} (needs Internet ON)...")
    DATA_CACHE.mkdir(parents=True, exist_ok=True)
    if name == "mvtec":
        try:
            import kagglehub
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
            import kagglehub
        p = kagglehub.dataset_download("ipythonx/mvtec-ad")
        root = find_dataset_root("mvtec", search=[str(p)])
    else:
        tar = DATA_CACHE / "VisA_20220922.tar"
        out = DATA_CACHE / "VisA"
        if not (out / ".extracted").exists():
            if not tar.exists():
                url = "https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar"
                subprocess.run(["wget", "-q", "-O", str(tar), url], check=True)
            out.mkdir(parents=True, exist_ok=True)
            subprocess.run(["tar", "-xf", str(tar), "-C", str(out)], check=True)
            (out / ".extracted").touch()
            tar.unlink(missing_ok=True)
        root = find_dataset_root("visa", search=[str(out)])
    print(f"[{name}] RGB images ready at {root}")
    return root


BUNDLE = locate_official_bundle()
GT_ROOT = BUNDLE["gt_root"]
FIT_R_DIR = BUNDLE["fit_r_dir"]
FIT_T_DIR = BUNDLE["fit_t_dir"]
ALL_RUNS_CSV = BUNDLE["all_runs_csv"]
PER_ITEM_ALL = BUNDLE["per_item_all"]

print(f"Official bundle   : {BUNDLE['parent']}")
print(f"Cached GT masks   : {GT_ROOT}")
print(f"Regime R maps dir : {FIT_R_DIR}")
print(f"Regime T maps dir : {FIT_T_DIR}")
print(f"all_runs.csv      : {ALL_RUNS_CSV}")
print(f"per_item_all      : {PER_ITEM_ALL}")
print(f"Output assets dir : {OUT_DIR.resolve()}")


## 2. Official Post-Processing, Cached Native Masks & Exact Fast `672×672` Metrics

In [ ]:
# ============================ 2. POST-PROCESSING & FAST EXACT 672x672 METRICS ============================
import torch
import torch.nn.functional as F

DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
_trapz = getattr(np, "trapezoid", None) or np.trapz
NAN = float("nan")
print(f"Running on device: {DEV} (0 GPU forward passes needed)")


def load_gt_meta(ds: str, cat: str) -> Dict[str, Any]:
    return json.loads((GT_ROOT / ds / cat / "testset.json").read_text())


def load_gt_masks_cached(ds: str, cat: str, size: int) -> np.ndarray:
    """Load native binary masks directly from cached gt/<ds>/<cat>/masks.npz and resize with NEAREST."""
    meta = load_gt_meta(ds, cat)
    n = len(meta["labels"])
    with np.load(GT_ROOT / ds / cat / "masks.npz") as z:
        files = set(z.files)
        out = []
        for i in range(n):
            key = f"m{i}"
            if key in files:
                arr = z[key]
                if arr.shape == (size, size):
                    out.append(arr.astype(bool, copy=False))
                else:
                    m = Image.fromarray(arr.astype(np.uint8) * 255).resize((size, size), Image.NEAREST)
                    out.append(np.asarray(m) > 0)
            else:
                out.append(np.zeros((size, size), dtype=bool))
    return np.stack(out, axis=0)


def load_regime_t_maps(ds: str, cat: str) -> np.ndarray:
    """Load Regime T maps safely as (n_test, G, G) regardless of 4D (n, 1, G, G) or 3D (n, G, G) storage."""
    arr = np.load(FIT_T_DIR / ds / cat / "maps.npy")
    return arr[:, 0] if arr.ndim == 4 else arr


def _gauss_kernel(ksize: int, sigma: float, dev: torch.device) -> torch.Tensor:
    xs = torch.arange(ksize, dtype=torch.float64, device=dev) - (ksize - 1) / 2.0
    k = torch.exp(-(xs ** 2) / (2.0 * sigma ** 2))
    return (k / k.sum()).float()


@torch.inference_mode()
def postprocess_maps(maps: np.ndarray, out_size: int = IMG_SIZE, batch_size: int = 8) -> Tuple[np.ndarray, np.ndarray]:
    """Exact official SubspaceAD post-processing: bilinear -> 672 -> 3x3 Gaussian (sigma=4, reflect-101) -> mtop1p."""
    S, ks, sig = int(IMG_SIZE), int(BLUR_K), float(BLUR_SIGMA)
    k = _gauss_kernel(ks, sig, DEV)
    kh, kv, r = k.view(1, 1, 1, -1), k.view(1, 1, -1, 1), ks // 2
    top_k = max(1, int(S * S * float(TOP_PCT) / 100.0))

    img_sc, ups = [], []
    for i in range(0, len(maps), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(maps[i : i + batch_size], dtype=np.float32)).to(DEV).unsqueeze(1)
        x = F.interpolate(x, size=(S, S), mode="bilinear", align_corners=False)
        if r > 0:
            x = F.pad(x, (r, r, r, r), mode="reflect")
            x = F.conv2d(F.conv2d(x, kh), kv)
        img_sc.append(x.flatten(1).topk(top_k, dim=1).values.mean(1).cpu())
        if int(out_size) != S:
            x = F.interpolate(x, size=(int(out_size), int(out_size)), mode="bilinear", align_corners=False, antialias=True)
        ups.append(x.squeeze(1).cpu().numpy().astype(np.float32))
    return torch.cat(img_sc).numpy(), np.concatenate(ups, axis=0)


def rank_metrics(pos_sorted: np.ndarray, neg_sorted: np.ndarray) -> Tuple[float, float, float]:
    """Exact pixel AUROC (ties=1/2), AP, and oracle max-F1 from sorted pos/neg scores (identical to sklearn)."""
    P, N = len(pos_sorted), len(neg_sorted)
    if P == 0 or N == 0:
        return NAN, NAN, NAN
    lo = np.searchsorted(neg_sorted, pos_sorted, side="left").sum(dtype=np.float64)
    hi = np.searchsorted(neg_sorted, pos_sorted, side="right").sum(dtype=np.float64)
    auroc = (lo + hi) / (2.0 * P * N)
    thr, first = np.unique(pos_sorted, return_index=True)
    n_at = np.diff(np.append(first, P)).astype(np.float64)
    tp = (P - first).astype(np.float64)
    fp = (N - np.searchsorted(neg_sorted, thr, side="left")).astype(np.float64)
    ap = float(np.sum(n_at / P * tp / (tp + fp)))
    f1 = float(np.max(2.0 * tp / (tp + fp + P)))
    return float(auroc), ap, f1


def precompute_pro_index(masks: np.ndarray) -> Optional[Dict[str, Any]]:
    """Precompute 8-connected GT region coordinates ONCE per category so 1/2/4-shot + T share it."""
    if not masks.any():
        return None
    structure = ndimage.generate_binary_structure(2, 2)
    slices: List[Tuple[int, np.ndarray, np.ndarray]] = []
    for i in np.flatnonzero(masks.reshape(len(masks), -1).any(1)):
        lab, n = ndimage.label(masks[i], structure=structure)
        for rid in range(1, n + 1):
            ys, xs = np.where(lab == rid)
            slices.append((int(i), ys.astype(np.int32), xs.astype(np.int32)))
    return dict(bg_mask=~masks, fg_mask=masks, slices=slices)


def aupro_from_sorted(neg_sorted: np.ndarray, regions: List[np.ndarray], fpr_limit: float, num_thresholds: int = PRO_N_THR) -> float:
    """Exact port of official SubspaceAD compute_aupro (main.py)."""
    if not regions or neg_sorted.size == 0:
        return NAN
    n_neg = neg_sorted.size
    target = np.linspace(0.0, fpr_limit, num_thresholds + 1)[1:]
    q_idx = np.clip(np.floor((1.0 - target) * (n_neg - 1)).astype(np.int64), 0, n_neg - 1)
    thr = neg_sorted[q_idx]
    fprs = (n_neg - np.searchsorted(neg_sorted, thr, side="left")).astype(np.float64) / n_neg
    pros = np.zeros(num_thresholds, dtype=np.float64)
    for rs in regions:
        pros += (rs.size - np.searchsorted(rs, thr, side="left")) / rs.size
    pros /= len(regions)
    order = np.argsort(fprs, kind="stable")
    f = np.concatenate([[0.0], fprs[order]])
    p = np.concatenate([[0.0], pros[order]])
    if f[-1] > fpr_limit:
        cut = np.searchsorted(f, fpr_limit, side="right")
        f0, f1, p0, p1 = f[cut - 1], f[cut], p[cut - 1], p[cut]
        p_at = p0 + (p1 - p0) * (fpr_limit - f0) / (f1 - f0) if f1 > f0 else p0
        f, p = np.concatenate([f[:cut], [fpr_limit]]), np.concatenate([p[:cut], [p_at]])
    elif f[-1] < fpr_limit:
        f, p = np.concatenate([f, [fpr_limit]]), np.concatenate([p, [p[-1]]])
    return float(_trapz(p, f) / fpr_limit)


def pixel_metrics_fast(amaps: np.ndarray, precomp: Optional[Dict[str, Any]]) -> Dict[str, float]:
    """Compute all 5 exact pixel metrics using a SINGLE sort of background pixels."""
    out = dict(pix_auroc=NAN, pix_ap=NAN, pix_f1_max_oracle=NAN, aupro_30=NAN, aupro_5=NAN)
    if precomp is None or not np.isfinite(amaps).all():
        return out
    neg_sorted = np.sort(amaps[precomp["bg_mask"]].astype(np.float32, copy=False))
    pos_sorted = np.sort(amaps[precomp["fg_mask"]].astype(np.float32, copy=False))
    out["pix_auroc"], out["pix_ap"], out["pix_f1_max_oracle"] = rank_metrics(pos_sorted, neg_sorted)
    regions = [np.sort(amaps[i, ys, xs].astype(np.float32, copy=False)) for i, ys, xs in precomp["slices"]]
    out["aupro_30"] = aupro_from_sorted(neg_sorted, regions, 0.30)
    out["aupro_5"] = aupro_from_sorted(neg_sorted, regions, 0.05)
    return out


def _self_test_metrics():
    rng = np.random.RandomState(0)
    y = rng.rand(3000) < 0.1
    s = np.round(rng.randn(3000) + 1.5 * y, 1)
    a, ap, f1 = rank_metrics(np.sort(s[y]), np.sort(s[~y]))
    pr, rc, _ = precision_recall_curve(y, s)
    f1_ref = np.max(np.where(pr + rc > 0, 2 * pr * rc / np.maximum(pr + rc, 1e-12), 0))
    assert abs(a - roc_auc_score(y, s)) < 1e-12
    assert abs(ap - average_precision_score(y, s)) < 1e-12
    assert abs(f1 - f1_ref) < 1e-12
    m = np.zeros((2, 64, 64), bool)
    m[0, 6:18, 6:18] = True
    m[1, 30:44, 30:44] = True
    sc = np.where(m, 1.0, 0.0).astype(np.float32) + rng.uniform(0, 1e-3, m.shape).astype(np.float32)
    pm = pixel_metrics_fast(sc, precompute_pro_index(m))
    assert pm["aupro_30"] > 0.995 and pm["aupro_5"] > 0.995
    print("[ok] Fast exact 672x672 pixel metrics verified (< 1e-12 vs sklearn + official AUPRO sanity).")

_self_test_metrics()


## 3. Publication-Quality Benchmark & Calibration Figures (`300 DPI`, Zero Label Collisions)

In [ ]:
# ============================ 3. PUBLICATION-QUALITY BENCHMARK FIGURES ============================
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def load_all_runs_df() -> pd.DataFrame:
    if ALL_RUNS_CSV is not None and ALL_RUNS_CSV.exists():
        return pd.read_csv(ALL_RUNS_CSV)
    rows = []
    for base_dir in [FIT_R_DIR, FIT_T_DIR]:
        for f in sorted(base_dir.glob("*/*/result_*.json")):
            r = json.loads(f.read_text())
            for run in r["runs"]:
                rows.append(dict(dataset=r["dataset"], category=r["category"], **run))
    return pd.DataFrame(rows)


def load_per_item_all_df() -> pd.DataFrame:
    if PER_ITEM_ALL is not None and PER_ITEM_ALL.exists():
        return pd.read_parquet(PER_ITEM_ALL) if PER_ITEM_ALL.suffix == ".parquet" else pd.read_csv(PER_ITEM_ALL)
    dfs = []
    for base_dir in [FIT_R_DIR, FIT_T_DIR]:
        for f in sorted(base_dir.glob("*/*/per_item_*.parquet")):
            dfs.append(pd.read_parquet(f))
        if not dfs:
            for f in sorted(base_dir.glob("*/*/per_item_*.csv")):
                dfs.append(pd.read_csv(f))
    return pd.concat(dfs, ignore_index=True)


df_all = load_all_runs_df()
df_items = load_per_item_all_df()
SCORE_COL = "image_score" if "image_score" in df_items.columns else "image_score_raw"
datasets_present = [d for d in ["mvtec", "visa"] if d in set(df_all["dataset"])]
print(f"Loaded {len(df_all)} run rows and {len(df_items):,} per-image score rows (score_col='{SCORE_COL}').")

# --- Figure 1: Few-shot scaling curve (clean smart label offsets, zero collisions!) ---
fig, axes = plt.subplots(1, len(datasets_present), figsize=(6.8 * len(datasets_present), 4.8), squeeze=False)
for idx_d, ds in enumerate(datasets_present):
    ax = axes[0, idx_d]
    sub = df_all[df_all["dataset"] == ds]
    sub_r = sub[sub["regime"] == "R"]
    sub_t = sub[sub["regime"] == "T"]

    per_seed = sub_r.groupby(["shot", "seed"])["img_auroc"].mean() * 100.0
    stats = per_seed.groupby("shot").agg(["mean", "std"]).reset_index()
    shots = stats["shot"].values
    means = stats["mean"].values
    stds = stats["std"].fillna(0.0).values

    ax.errorbar(shots, means, yerr=stds, fmt="-o", linewidth=2.3, markersize=7.5, capsize=5,
                color="#1d4ed8", zorder=4, label="Ours — Regime R (5-seed mean ± std)")

    ref_r = REF_IMG_AUROC_R.get(ds, {})
    if ref_r:
        ref_x = sorted(ref_r.keys())
        ref_y = [ref_r[k] for k in ref_x]
        ax.plot(ref_x, ref_y, "s--", color="#15803d", linewidth=1.7, markersize=6, zorder=3,
                label="Official CVPR'26 README (Regime R)")

    t_val = sub_t["img_auroc"].mean() * 100.0 if len(sub_t) else None
    if t_val is not None:
        ax.axhline(t_val, color="#dc2626", linestyle="-.", linewidth=1.8, zorder=2,
                   label=f"Ours — Regime T 0-shot ({t_val:.2f}%)")
    if ds in REF_IMG_AUROC_T:
        ax.axhline(REF_IMG_AUROC_T[ds], color="#ea580c", linestyle=":", linewidth=2.0, zorder=2,
                   label=f"Official README — Regime T ({REF_IMG_AUROC_T[ds]:.1f}%)")

    # Smart non-overlapping annotation placement
    for x, y, sd in zip(shots, means, stds):
        # Put label below point if a reference point or Regime T line sits right above it
        above_crowded = (
            (int(x) in ref_r and 0.0 <= (ref_r[int(x)] - y) < 0.7)
            or (t_val is not None and 0.0 <= (t_val - y) < 0.5)
        )
        dy = -(16 + int(sd * 8)) if above_crowded else (11 + int(sd * 5))
        va = "top" if above_crowded else "bottom"
        ax.annotate(
            f"{y:.2f}%",
            (x, y),
            textcoords="offset points",
            xytext=(0, dy),
            ha="center",
            va=va,
            fontsize=9.2,
            fontweight="bold",
            color="#1e3a8a",
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor="none", alpha=0.82),
        )

    ax.set_title(f"{ds.upper()} — Image AUROC (%) vs Few-Shot Count (Official Final)", fontweight="bold")
    ax.set_xlabel("Normal Reference Images per Category (k-shot)")
    ax.set_ylabel("Image AUROC (%)")
    ax.set_xticks([1, 2, 4])
    ax.set_xlim(0.55, 4.45)
    ymin = min(means.min() - stds.max() - 1.3, (REF_IMG_AUROC_T.get(ds, 95) - 1.8))
    ymax = max(means.max() + stds.max() + 1.4, (max(ref_r.values()) + 1.2 if ref_r else 100.0))
    ax.set_ylim(max(85.0, ymin), min(100.6, ymax))
    ax.legend(loc="lower right", fontsize=8.4, frameon=True, facecolor="white", framealpha=0.92)

plt.tight_layout()
fig.savefig(FIG_DIR / "fig1_fewshot_scaling.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# --- Figure 2: Per-category Regime R (4-shot) vs Regime T (0-shot) comparison ---
for ds in datasets_present:
    sub = df_all[df_all["dataset"] == ds]
    r4 = sub[(sub["regime"] == "R") & (sub["shot"] == 4)].groupby("category")[["img_auroc", "aupro_30"]].mean() * 100.0
    t0 = sub[sub["regime"] == "T"].groupby("category")[["img_auroc", "aupro_30"]].mean() * 100.0
    cats = sorted(r4.index)
    y_pos = np.arange(len(cats))
    h = 0.36

    fig, axes = plt.subplots(1, 2, figsize=(13.2, max(4.8, 0.44 * len(cats))), sharey=True)
    for ax, metric, title in zip(axes, ["img_auroc", "aupro_30"], ["Image AUROC (%)", "AUPRO@30% (%)"]):
        v_r4 = [r4.loc[c, metric] for c in cats]
        v_t0 = [t0.loc[c, metric] if c in t0.index else np.nan for c in cats]
        ax.barh(y_pos + h / 2, v_r4, height=h, color="#2563eb", alpha=0.92, label="Regime R (4-shot, 5-seed mean)")
        ax.barh(y_pos - h / 2, v_t0, height=h, color="#ea580c", alpha=0.88, label="Regime T (Batched 0-shot)")
        for i, (vr, vt) in enumerate(zip(v_r4, v_t0)):
            ax.text(vr + 0.35, i + h / 2, f"{vr:.1f}", va="center", fontsize=8.2, color="#1e40af")
            if np.isfinite(vt):
                ax.text(vt + 0.35, i - h / 2, f"{vt:.1f}", va="center", fontsize=8.2, color="#9a3412")
        ax.set_title(f"{ds.upper()} — {title}", fontweight="bold")
        ax.set_xlabel(title)
        lo_x = max(55.0, min(np.nanmin(v_r4), np.nanmin(v_t0)) - 6.0)
        ax.set_xlim(lo_x, 104.0)
        ax.legend(loc="lower left", fontsize=8.5, frameon=True)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(cats)
    axes[0].invert_yaxis()
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"fig2_{ds}_per_category_R4_vs_T.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# --- Figure 3: Normal vs Anomalous Score Distributions (Regime R 4-shot s0) ---
hist_cats = [
    c for c in [("mvtec", "bottle"), ("mvtec", "capsule"), ("mvtec", "transistor"), ("visa", "pcb3")]
    if len(df_items[(df_items["dataset"] == c[0]) & (df_items["category"] == c[1])]) > 0
]
if hist_cats:
    fig, axes = plt.subplots(1, len(hist_cats), figsize=(4.3 * len(hist_cats), 3.7), squeeze=False)
    for idx_c, (ds, cat) in enumerate(hist_cats):
        ax = axes[0, idx_c]
        sub_pi = df_items[
            (df_items["dataset"] == ds) & (df_items["category"] == cat)
            & (df_items["regime"] == "R") & (df_items["shot"] == 4) & (df_items["seed"] == 0)
        ]
        norm_s = sub_pi[sub_pi["label"] == 0][SCORE_COL].values
        anom_s = sub_pi[sub_pi["label"] == 1][SCORE_COL].values
        hi = np.percentile(np.concatenate([norm_s, anom_s]), 99)
        bins = np.linspace(min(norm_s.min(), anom_s.min()), hi, 22)
        ax.hist(norm_s, bins=bins, alpha=0.65, color="#16a34a", label=f"Normal (n={len(norm_s)})", density=True)
        ax.hist(anom_s, bins=bins, alpha=0.60, color="#dc2626", label=f"Anomalous (n={len(anom_s)})", density=True)
        q95 = np.quantile(norm_s, 0.95)
        ax.axvline(q95, color="#15803d", linestyle="--", linewidth=1.4, label=f"Norm q95={q95:.1f}")
        auc = roc_auc_score(sub_pi["label"].values, sub_pi[SCORE_COL].values) * 100.0
        ax.set_title(f"{ds}/{cat} (4-shot s0)\nImg AUROC = {auc:.2f}%", fontsize=9.5, fontweight="bold")
        ax.set_xlabel("Raw Image Score (mtop1p)")
        if idx_c == 0:
            ax.set_ylabel("Density")
        ax.legend(fontsize=7.8)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "fig3_score_distributions_calibration.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# --- Figure 4: Cross-Category & Cross-Regime Raw Score Scale Mismatch (Calibration Justification) ---
sub_r4_s0 = df_items[(df_items["regime"] == "R") & (df_items["shot"] == 4) & (df_items["seed"] == 0) & (df_items["label"] == 0)]
sub_t0_n = df_items[(df_items["regime"] == "T") & (df_items["label"] == 0)]
if len(sub_r4_s0) and len(sub_t0_n):
    g_r = sub_r4_s0.groupby(["dataset", "category"])[SCORE_COL].agg(med="median", q95=lambda x: np.quantile(x, 0.95))
    g_t = sub_t0_n.groupby(["dataset", "category"])[SCORE_COL].agg(med="median", q95=lambda x: np.quantile(x, 0.95))
    keys = sorted(g_r.index)
    labels_x = [f"{d[:1].upper()}:{c}" for d, c in keys]
    x = np.arange(len(keys))
    w = 0.38

    fig, ax = plt.subplots(figsize=(max(11, 0.52 * len(keys)), 4.6))
    ax.bar(x - w / 2, [g_r.loc[k, "q95"] for k in keys], width=w, color="#2563eb", alpha=0.88,
           label="Regime R (4-shot s0) — Normal 95th Percentile Raw Score")
    ax.bar(x + w / 2, [g_t.loc[k, "q95"] if k in g_t.index else np.nan for k in keys], width=w, color="#ea580c", alpha=0.88,
           label="Regime T (0-shot) — Normal 95th Percentile Raw Score")
    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, rotation=50, ha="right", fontsize=8.5)
    ax.set_ylabel("Raw Score (mtop1p) at Normal q95")
    ax.set_title("Why Per-Category Calibration Is Essential: Raw Normal Score Scale Varies >5x Across Categories & Regimes",
                 fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "fig4_regime_scale_mismatch.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print(f"Saved benchmark & calibration charts to: {FIG_DIR}")


## 4. Qualitative Heatmaps & Hard/Failure-Case Inspection (`300 DPI`)

In [ ]:
# ============================ 4. QUALITATIVE HEATMAPS & HARD/FAILURE CASES ============================
DS_ROOTS: Dict[str, Optional[Path]] = {
    "mvtec": ensure_dataset_opt("mvtec"),
    "visa": ensure_dataset_opt("visa") if VISA_RGB_VISUALS else None,
}


def make_overlay(img_rgb: np.ndarray, heat01: np.ndarray, alpha: float = 0.48) -> np.ndarray:
    cmap = matplotlib.colormaps["jet"]
    colored = cmap(np.clip(heat01, 0.0, 1.0))[..., :3]
    return np.clip((1.0 - alpha) * img_rgb + alpha * colored, 0.0, 1.0)


def draw_gt_with_contour(img_rgb: np.ndarray, mask: np.ndarray) -> np.ndarray:
    vis = img_rgb.copy()
    if mask.any():
        vis[mask] = 0.55 * vis[mask] + 0.45 * np.array([1.0, 0.15, 0.15])
        edge = ndimage.binary_dilation(mask, iterations=2) ^ mask
        vis[edge] = np.array([1.0, 0.95, 0.1])
    return np.clip(vis, 0.0, 1.0)


def plot_category_diagnostic_sheet(
    ds: str,
    cat: str,
    shot: int = 4,
    seed: int = 0,
    n_typical_anom: int = 3,
    disp_size: int = 384,
) -> Optional[Path]:
    """Generate a clean diagnostic figure: 3 defects + hardest FN (min score) + hardest FP (max normal)."""
    root = DS_ROOTS.get(ds)
    if root is None:
        print(f"[skip heatmap] {ds}/{cat}: RGB dataset images not available.")
        return None
    fdir_r = FIT_R_DIR / ds / cat
    fdir_t = FIT_T_DIR / ds / cat
    if not (fdir_r / "maps.npy").exists() or not (fdir_t / "maps.npy").exists():
        print(f"[skip heatmap] {ds}/{cat}: cached maps not found.")
        return None

    gt_meta = load_gt_meta(ds, cat)
    meta_r = json.loads((fdir_r / "meta.json").read_text())
    maps_r = np.load(fdir_r / "maps.npy")
    raw_t = load_regime_t_maps(ds, cat)
    masks_disp = load_gt_masks_cached(ds, cat, disp_size)

    j = [tuple(int(v) for v in c) for c in meta_r["cfgs"]].index((int(shot), int(seed)))
    raw_r = maps_r[:, j]

    sc_r, up_r = postprocess_maps(raw_r, out_size=disp_size)
    sc_t, up_t = postprocess_maps(raw_t, out_size=disp_size)
    lo_r, hi_r = np.percentile(up_r, 1.0), np.percentile(up_r, 99.9)
    lo_t, hi_t = np.percentile(up_t, 1.0), np.percentile(up_t, 99.9)

    labels = np.array(gt_meta["labels"], dtype=int)
    anom_idx = np.where(labels == 1)[0]
    norm_idx = np.where(labels == 0)[0]

    anom_sorted = anom_idx[np.argsort(sc_r[anom_idx])]
    hardest_fn = int(anom_sorted[0])
    other_anom = anom_sorted[1:]
    if len(other_anom) >= n_typical_anom:
        pick_pos = np.linspace(len(other_anom) - 1, max(0, len(other_anom) // 3), n_typical_anom).astype(int)
        typical = [int(other_anom[p]) for p in pick_pos]
    else:
        typical = [int(i) for i in other_anom]
    hardest_fp = int(norm_idx[np.argmax(sc_r[norm_idx])]) if len(norm_idx) else None

    rows_spec: List[Tuple[int, str]] = [(idx, f"Defect #{k+1}") for k, idx in enumerate(typical)]
    rows_spec.append((hardest_fn, "Hardest Defect (Min Score)"))
    if hardest_fp is not None:
        rows_spec.append((hardest_fp, "Hardest Normal (Max FP)"))

    n_rows = len(rows_spec)
    fig, axes = plt.subplots(n_rows, 4, figsize=(13.5, 3.25 * n_rows), squeeze=False)
    col_titles = [
        "Original Test Image",
        "Ground Truth Defect",
        f"Regime R ({shot}-shot, s{seed})",
        "Regime T (0-shot)",
    ]
    for c_i, title in enumerate(col_titles):
        axes[0, c_i].set_title(title, fontsize=11, fontweight="bold", pad=8)

    for r_i, (img_i, row_tag) in enumerate(rows_spec):
        rel_p = gt_meta["rel_paths"][img_i]
        img = np.asarray(Image.open(root / rel_p).convert("RGB").resize((disp_size, disp_size), Image.BICUBIC), dtype=np.float32) / 255.0
        msk = masks_disp[img_i]

        h_r = np.clip((up_r[img_i] - lo_r) / (hi_r - lo_r + 1e-9), 0.0, 1.0)
        h_t = np.clip((up_t[img_i] - lo_t) / (hi_t - lo_t + 1e-9), 0.0, 1.0)

        axes[r_i, 0].imshow(img)
        axes[r_i, 1].imshow(draw_gt_with_contour(img, msk))
        axes[r_i, 2].imshow(make_overlay(img, h_r))
        axes[r_i, 3].imshow(make_overlay(img, h_t))

        defect_folder = Path(rel_p).parent.name
        axes[r_i, 0].set_ylabel(f"{row_tag}\n({defect_folder})", fontsize=9.5, fontweight="bold")
        axes[r_i, 2].set_xlabel(f"score = {sc_r[img_i]:.1f}", fontsize=9)
        axes[r_i, 3].set_xlabel(f"score = {sc_t[img_i]:.1f}", fontsize=9)

        for c_i in range(4):
            axes[r_i, c_i].set_xticks([])
            axes[r_i, c_i].set_yticks([])

    auc_r = roc_auc_score(labels, sc_r) * 100.0
    auc_t = roc_auc_score(labels, sc_t) * 100.0
    fig.suptitle(
        f"SubspaceAD Official Final — {ds.upper()} / {cat}  |  "
        f"Img AUROC: R({shot}-shot s{seed})={auc_r:.2f}%  vs  T(0-shot)={auc_t:.2f}%",
        fontsize=12.5, fontweight="bold", y=0.992,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    out_png = HEATMAP_DIR / f"heatmap_{ds}_{cat}.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return out_png


saved_heatmaps = []
for ds, cat in SHOW_CATS:
    p = plot_category_diagnostic_sheet(ds, cat)
    if p is not None:
        saved_heatmaps.append(p)
print(f"Generated {len(saved_heatmaps)} diagnostic heatmap sheets in: {HEATMAP_DIR}")


## 5. DefectLens Implementation-Phase Assets: Calibration Tables & VLM ROI Crop Proposals

In [ ]:
# ============================ 5. IMPLEMENTATION-PHASE ASSETS (CALIBRATION + VLM ROI PROPOSALS) ============================
# 5A. Per-(dataset, category, regime, shot, seed) calibration & conformal statistics table
cal_rows = []
for (ds, cat, rg, shot, seed), g in df_items.groupby(["dataset", "category", "regime", "shot", "seed"]):
    y = g["label"].values.astype(int)
    s = g[SCORE_COL].values.astype(np.float64)
    norm_s = s[y == 0]
    anom_s = s[y == 1]
    if len(norm_s) == 0 or len(anom_s) == 0:
        continue

    q25, q50, q75 = np.quantile(norm_s, [0.25, 0.50, 0.75])
    q90, q95, q99 = np.quantile(norm_s, [0.90, 0.95, 0.99])
    iqr = float(max(q75 - q25, 1e-6))

    prec, rec, pr_thr = precision_recall_curve(y, s)
    f1_arr = np.where(prec + rec > 0, 2.0 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)
    best_i = int(np.argmax(f1_arr))
    thr_f1 = float(pr_thr[min(best_i, len(pr_thr) - 1)]) if len(pr_thr) else float("nan")

    fpr, tpr, roc_thr = roc_curve(y, s)
    idx_fpr1 = np.where(fpr <= 0.01)[0]
    idx_fpr5 = np.where(fpr <= 0.05)[0]
    thr_fpr1 = float(roc_thr[idx_fpr1[-1]]) if len(idx_fpr1) else float("nan")
    tpr_fpr1 = float(tpr[idx_fpr1[-1]]) if len(idx_fpr1) else float("nan")
    thr_fpr5 = float(roc_thr[idx_fpr5[-1]]) if len(idx_fpr5) else float("nan")
    tpr_fpr5 = float(tpr[idx_fpr5[-1]]) if len(idx_fpr5) else float("nan")

    cal_rows.append(dict(
        dataset=ds, category=cat, regime=rg, shot=int(shot), seed=int(seed),
        n_normal=int(len(norm_s)), n_anomalous=int(len(anom_s)),
        img_auroc=round(float(roc_auc_score(y, s)) * 100.0, 2),
        norm_mean=round(float(norm_s.mean()), 4),
        norm_std=round(float(norm_s.std(ddof=0)), 4),
        norm_median=round(float(q50), 4),
        norm_iqr=round(iqr, 4),
        norm_q90=round(float(q90), 4),
        norm_q95=round(float(q95), 4),
        norm_q99=round(float(q99), 4),
        norm_max=round(float(norm_s.max()), 4),
        anom_min=round(float(anom_s.min()), 4),
        anom_median=round(float(np.median(anom_s)), 4),
        anom_max=round(float(anom_s.max()), 4),
        separation_gap=round(float(anom_s.min() - norm_s.max()), 4),
        n_anom_below_norm_q95=int((anom_s <= q95).sum()),
        thr_fpr1=round(thr_fpr1, 4), tpr_at_fpr1=round(tpr_fpr1 * 100.0, 2),
        thr_fpr5=round(thr_fpr5, 4), tpr_at_fpr5=round(tpr_fpr5 * 100.0, 2),
        thr_oracle_f1=round(thr_f1, 4), oracle_f1=round(float(f1_arr[best_i]) * 100.0, 2),
    ))

df_cal = pd.DataFrame(cal_rows)
df_cal.to_csv(IMPL_DIR / "subspacead_calibration_stats.csv", index=False)
print(f"[ok] Saved calibration statistics for {len(df_cal)} runs -> {IMPL_DIR / 'subspacead_calibration_stats.csv'}")

# 5B. Per-image VLM ROI Bounding Box Proposals (Regime R 4-shot s0 + Regime T 0-shot, all 27 categories)
def extract_peak_roi_bbox(amap: np.ndarray, pad_frac: float = 0.08, min_box_frac: float = 0.16) -> Tuple[int, int, int, int, float, float]:
    """Extract padded ROI bbox (x1, y1, x2, y2) around the peak anomaly component using half-max above background."""
    H, W = amap.shape
    peak = float(amap.max())
    bg = float(np.median(amap))
    q25, q75 = np.percentile(amap, [25.0, 75.0])
    iqr = float(max(q75 - q25, 1e-6))
    # Half-maximum above background captures full extent of both small and large defects without clipping
    thr = bg + 0.50 * (peak - bg) if (peak - bg) > 3.0 * iqr else float(np.percentile(amap, 98.5))
    hot = amap >= thr
    if not hot.any():
        py, px = np.unravel_index(int(np.argmax(amap)), (H, W))
        hot[py, px] = True

    lab, n_cc = ndimage.label(hot, structure=np.ones((3, 3)))
    # Pick the connected component containing the global peak
    py, px = np.unravel_index(int(np.argmax(amap)), (H, W))
    best_id = int(lab[py, px])
    comp = (lab == best_id) if best_id > 0 else hot
    ys, xs = np.where(comp)

    y1, y2 = int(ys.min()), int(ys.max())
    x1, x2 = int(xs.min()), int(xs.max())
    pad_y = int(round(H * pad_frac))
    pad_x = int(round(W * pad_frac))
    y1, y2 = max(0, y1 - pad_y), min(H - 1, y2 + pad_y)
    x1, x2 = max(0, x1 - pad_x), min(W - 1, x2 + pad_x)

    # Enforce minimum crop size so tiny pinpoint defects have visual context for the VLM
    min_h, min_w = int(round(H * min_box_frac)), int(round(W * min_box_frac))
    if (y2 - y1) < min_h:
        cy = (y1 + y2) // 2
        y1 = max(0, min(H - min_h, cy - min_h // 2))
        y2 = min(H - 1, y1 + min_h)
    if (x2 - x1) < min_w:
        cx = (x1 + x2) // 2
        x1 = max(0, min(W - min_w, cx - min_w // 2))
        x2 = min(W - 1, x1 + min_w)

    area_frac = float(((x2 - x1 + 1) * (y2 - y1 + 1)) / float(H * W))
    return x1, y1, x2, y2, area_frac, peak


roi_rows = []
t_roi = time.time()
for ds in datasets_present:
    for cat in CATS[ds]:
        fdir_r = FIT_R_DIR / ds / cat
        fdir_t = FIT_T_DIR / ds / cat
        if not (fdir_r / "maps.npy").exists():
            continue
        gt_meta = load_gt_meta(ds, cat)
        labels = np.array(gt_meta["labels"], dtype=int)
        rel_paths = gt_meta["rel_paths"]

        meta_r = json.loads((fdir_r / "meta.json").read_text())
        j_r4 = [tuple(int(v) for v in c) for c in meta_r["cfgs"]].index((4, 0))
        regimes_spec = [("R", 4, 0, np.load(fdir_r / "maps.npy")[:, j_r4])]
        if (fdir_t / "maps.npy").exists():
            regimes_spec.append(("T", 0, 0, load_regime_t_maps(ds, cat)))

        for rg, sh, sd, raw_m in regimes_spec:
            sc, up = postprocess_maps(raw_m, out_size=256)
            norm_s = sc[labels == 0]
            q25, q50, q75 = np.quantile(norm_s, [0.25, 0.50, 0.75])
            q90, q95, q99 = np.quantile(norm_s, [0.90, 0.95, 0.99])
            iqr = float(max(q75 - q25, 1e-6))

            for idx_i in range(len(labels)):
                x1, y1, x2, y2, area_frac, peak = extract_peak_roi_bbox(up[idx_i])
                s_val = float(sc[idx_i])
                z_robust = (s_val - float(q50)) / iqr
                if s_val < q90:
                    band = "auto_normal"
                elif s_val <= q99:
                    band = "borderline_send_to_vlm"
                else:
                    band = "high_conf_anomaly"
                roi_rows.append(dict(
                    dataset=ds, category=cat, regime=rg, shot=sh, seed=sd,
                    img_idx=idx_i, rel_path=rel_paths[idx_i], label=int(labels[idx_i]),
                    image_score=round(s_val, 4), map_peak=round(peak, 4),
                    norm_median=round(float(q50), 4), norm_q90=round(float(q90), 4),
                    norm_q95=round(float(q95), 4), norm_q99=round(float(q99), 4),
                    robust_z_score=round(float(z_robust), 3),
                    vlm_triage_band=band,
                    x1_norm=round(x1 / 256.0, 4), y1_norm=round(y1 / 256.0, 4),
                    x2_norm=round((x2 + 1) / 256.0, 4), y2_norm=round((y2 + 1) / 256.0, 4),
                    x1_672=int(round(x1 * IMG_SIZE / 256.0)), y1_672=int(round(y1 * IMG_SIZE / 256.0)),
                    x2_672=int(round((x2 + 1) * IMG_SIZE / 256.0)), y2_672=int(round((y2 + 1) * IMG_SIZE / 256.0)),
                    roi_area_frac=round(area_frac, 4),
                ))

df_roi = pd.DataFrame(roi_rows)
df_roi.to_csv(IMPL_DIR / "subspacead_vlm_roi_proposals.csv", index=False)
print(f"[ok] Extracted {len(df_roi):,} VLM ROI crop proposals across all categories in {time.time() - t_roi:.1f}s -> {IMPL_DIR / 'subspacead_vlm_roi_proposals.csv'}")
print("VLM Triage Band distribution (Regime R 4-shot s0):")
print(df_roi[df_roi["regime"] == "R"].groupby(["vlm_triage_band", "label"]).size().unstack(fill_value=0).to_string())

# 5C. Visual Preview of Extracted VLM ROI Crops on Hard Industrial Cases (fig5)
preview_items = []
for ds, cat in VLM_PREVIEW_CATS:
    root = DS_ROOTS.get(ds)
    if root is None:
        continue
    sub = df_roi[(df_roi["dataset"] == ds) & (df_roi["category"] == cat) & (df_roi["regime"] == "R")]
    if not len(sub):
        continue
    # Pick 1 clear defect (median anomalous score) + 1 borderline / hardest defect (min anomalous score)
    anoms = sub[sub["label"] == 1].sort_values("image_score")
    if len(anoms):
        preview_items.append((ds, cat, anoms.iloc[len(anoms) // 2], "Typical Defect ROI"))
        preview_items.append((ds, cat, anoms.iloc[0], "Hardest Defect ROI"))

if preview_items:
    n_rows = len(preview_items)
    fig, axes = plt.subplots(n_rows, 3, figsize=(10.8, 3.0 * n_rows), squeeze=False)
    for c_i, ttl in enumerate(["Full Image + Proposed VLM ROI Box", "Extracted VLM Zoom Crop", "Ground Truth Mask"]):
        axes[0, c_i].set_title(ttl, fontsize=10.5, fontweight="bold", pad=8)

    for r_i, (ds, cat, row, tag) in enumerate(preview_items):
        root = DS_ROOTS[ds]
        img = np.asarray(Image.open(root / row["rel_path"]).convert("RGB").resize((384, 384), Image.BICUBIC), dtype=np.float32) / 255.0
        msk = load_gt_masks_cached(ds, cat, 384)[int(row["img_idx"])]
        x1, y1 = int(round(row["x1_norm"] * 384)), int(round(row["y1_norm"] * 384))
        x2, y2 = max(x1 + 8, int(round(row["x2_norm"] * 384))), max(y1 + 8, int(round(row["y2_norm"] * 384)))

        axes[r_i, 0].imshow(img)
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.4, edgecolor="#ef4444", facecolor="none")
        axes[r_i, 0].add_patch(rect)
        axes[r_i, 0].set_ylabel(f"{ds}/{cat}\n{tag}", fontsize=9, fontweight="bold")
        axes[r_i, 0].set_xlabel(f"score={row['image_score']:.1f} | z={row['robust_z_score']:.1f} | {row['vlm_triage_band']}", fontsize=8.3)

        crop = img[y1:y2, x1:x2]
        axes[r_i, 1].imshow(crop)
        axes[r_i, 1].set_xlabel(f"ROI box ({row['x1_672']},{row['y1_672']})->({row['x2_672']},{row['y2_672']})", fontsize=8.3)

        axes[r_i, 2].imshow(draw_gt_with_contour(img, msk))
        for c_i in range(3):
            axes[r_i, c_i].set_xticks([])
            axes[r_i, c_i].set_yticks([])

    fig.suptitle("DefectLens Stage 1 -> Stage 3 Bridge: SubspaceAD Peak ROI Proposals for VLM Verifier",
                 fontsize=12, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(FIG_DIR / "fig5_vlm_roi_crop_preview.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 6. Official Full-Resolution `EVAL_SIZE = 672` Pixel Re-Evaluation (`0` GPU Forward Passes, From Cached `gt/masks.npz`)

In [ ]:
# ============================ 6. OFFICIAL 672x672 PIXEL RE-EVALUATION ============================
def evaluate_at_672(seed_mode: str = EVAL_672_SEEDS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Recompute all 5 exact pixel metrics at official 672x672 directly from cached maps.npy + gt/masks.npz."""
    rows_672 = []
    t_global = time.time()

    for ds in datasets_present:
        cats = [c for c in CATS[ds] if (FIT_R_DIR / ds / c / "maps.npy").exists() and (GT_ROOT / ds / c / "masks.npz").exists()]
        print(f"\n[672 eval] {ds.upper()}: evaluating {len(cats)} categories from cached native masks (seed_mode='{seed_mode}')...")

        for idx_c, cat in enumerate(cats, 1):
            t_cat = time.time()
            fdir_r = FIT_R_DIR / ds / cat
            fdir_t = FIT_T_DIR / ds / cat
            meta_r = json.loads((fdir_r / "meta.json").read_text())

            masks_672 = load_gt_masks_cached(ds, cat, IMG_SIZE)
            pro_pre = precompute_pro_index(masks_672)

            maps_r = np.load(fdir_r / "maps.npy")
            seeds_all = sorted({int(s) for _, s in meta_r["cfgs"]})
            keep_seeds = {seeds_all[0]} if seed_mode == "first" else set(seeds_all)

            for j, (k, s) in enumerate(meta_r["cfgs"]):
                if int(s) not in keep_seeds:
                    continue
                _, up_672 = postprocess_maps(maps_r[:, j], out_size=IMG_SIZE)
                pm = pixel_metrics_fast(up_672, pro_pre)
                rows_672.append(dict(
                    dataset=ds, category=cat, regime="R", shot=int(k), seed=int(s),
                    pix_auroc_672=pm["pix_auroc"], pix_ap_672=pm["pix_ap"],
                    pix_f1_max_oracle_672=pm["pix_f1_max_oracle"],
                    aupro_30_672=pm["aupro_30"], aupro_5_672=pm["aupro_5"],
                ))
            del maps_r

            if (fdir_t / "maps.npy").exists():
                raw_t = load_regime_t_maps(ds, cat)
                _, up_672 = postprocess_maps(raw_t, out_size=IMG_SIZE)
                pm = pixel_metrics_fast(up_672, pro_pre)
                rows_672.append(dict(
                    dataset=ds, category=cat, regime="T", shot=0, seed=0,
                    pix_auroc_672=pm["pix_auroc"], pix_ap_672=pm["pix_ap"],
                    pix_f1_max_oracle_672=pm["pix_f1_max_oracle"],
                    aupro_30_672=pm["aupro_30"], aupro_5_672=pm["aupro_5"],
                ))
                del raw_t

            del masks_672, pro_pre
            print(f"  [{idx_c:2d}/{len(cats)}] {ds}/{cat} done in {time.time() - t_cat:.1f}s")

    df_672 = pd.DataFrame(rows_672)
    df_672.to_csv(EVAL672_DIR / "eval672_per_run.csv", index=False)

    merge_cols = ["dataset", "category", "regime", "shot", "seed", "img_auroc", "pix_auroc", "pix_ap", "pix_f1_max_oracle", "aupro_30", "aupro_5"]
    avail_cols = [c for c in merge_cols if c in df_all.columns]
    merged = df_672.merge(df_all[avail_cols], on=["dataset", "category", "regime", "shot", "seed"], how="left")

    cmp_metrics = ["pix_auroc", "pix_auroc_672", "pix_ap", "pix_ap_672", "aupro_30", "aupro_30_672", "aupro_5", "aupro_5_672"]
    if "pix_f1_max_oracle" in merged.columns:
        cmp_metrics = ["pix_auroc", "pix_auroc_672", "pix_ap", "pix_ap_672", "pix_f1_max_oracle", "pix_f1_max_oracle_672", "aupro_30", "aupro_30_672", "aupro_5", "aupro_5_672"]

    comp = merged.groupby(["dataset", "regime", "shot"])[cmp_metrics].mean().mul(100.0).round(2)
    comp["d_pix_auroc"] = (comp["pix_auroc_672"] - comp["pix_auroc"]).round(2)
    comp["d_aupro_30"] = (comp["aupro_30_672"] - comp["aupro_30"]).round(2)
    comp["d_aupro_5"] = (comp["aupro_5_672"] - comp["aupro_5"]).round(2)
    comp.to_csv(EVAL672_DIR / "eval256_vs_eval672_comparison.csv")

    print(f"\nFinished official 672x672 evaluation in {(time.time() - t_global) / 60.0:.1f} min (0 GPU forward passes).")
    print("\n================ OFFICIAL FINAL: 256x256 vs 672x672 PIXEL METRICS (%) ================")
    print(comp.to_string())
    return df_672, comp


if RUN_EVAL_672:
    df_672, df_comp_256_672 = evaluate_at_672(EVAL_672_SEEDS)
else:
    print("RUN_EVAL_672=False -> skipped 672x672 pixel evaluation.")


## 7. Package All Report Figures, Heatmaps, `672×672` Tables & Implementation CSVs into One ZIP

In [ ]:
# ============================ 7. BUNDLE SINGLE DOWNLOADABLE ZIP ============================
zip_path = WORK / "subspacead_official_assets.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUT_DIR))

print(f"All report figures, heatmaps, 672x672 CSVs & implementation tables packaged into:\n  {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")
print("\nArchive contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print("  -", name)
